In [1]:
import os

In [2]:

import json
from IPython.display import Audio, display
import openai
from gtts import gTTS
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
openai.api_key = os.getenv('OPENAI_API_KEY')

In [4]:
# 1) STT: transcribe Urdu audio (convert speech to text)
def transcribe_audio(audio_path, model_preference=('whisper-1',)):
    """Transcribe audio file to Urdu script. Returns transcribed text."""
    with open(audio_path, 'rb') as af:
        resp = openai.audio.transcriptions.create(model=model_preference[0], file=af)
    return resp.text

In [ ]:
# 2) Convert Urdu-script to Roman-Urdu
def urdu_to_roman(urdu_text, model='gpt-4o-mini', temperature=0.0):
    sys_msg = 'You are a helpful assistant that converts Urdu (Arabic script) into Roman-Urdu (Latin characters). Preserve meaning, punctuation and numbers.'
    prompt = f'Convert the following Urdu text into Roman Urdu (use natural conversational transliteration).\n\nText:\n{urdu_text}\n\nReturn ONLY the Roman-Urdu text.'

    resp = openai.chat.completions.create(
        model=model,
        messages=[{'role': 'system', 'content': sys_msg}, {'role': 'user', 'content': prompt}],
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()

In [ ]:
# 3) Query the Chroma DataBase (path: ./chroma_db)
def generate_response(roman_query, chroma_persist_dir='chroma_db', collection_name='sample', model='gpt-4o-mini'):
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import Chroma

    emb = OpenAIEmbeddings()
    vect = Chroma(embedding_function=emb, persist_directory=chroma_persist_dir, collection_name=collection_name)
    retrieved = vect.similarity_search(roman_query, k=15)

    pieces = []
    for i, d in enumerate(retrieved):
        pieces.append(f'[{i+1}] Category: {d.metadata.get("category", "N/A")}\n{d.page_content}')
    context = '\n\n'.join(pieces)

    rag_prompt = f'''You are a helpful assistant for FAST-NUCES queries. Answer the user using ONLY the provided context. If the answer is not present in the context, clearly say you don't have enough information. Keep the answer concise and accurate. IMPORTANT FORMATTING RULES:- Write all numbers WITHOUT commas or separators.- For example: write 11000 instead of 11,000.- Do NOT use commas in currency values.- Always write currency like: Rs. 11000 per credit hour.
    Question: {roman_query}
    Context: {context}
    Provide a JSON object with keys "roman_urdu" and "arabic_urdu" containing the answer in Roman-Urdu and Urdu (Arabic script) respectively.
    '''
    resp = openai.chat.completions.create(
        model=model,
        messages=[{'role':'system','content':'You are a JSON-output generator.'},{'role':'user','content':rag_prompt}],
        temperature=0.1,
    )
    assistant_text = resp.choices[0].message.content
    try:
        start = assistant_text.find('{')
        end = assistant_text.rfind('}')
        return json.loads(assistant_text[start:end+1])
    except:
        return {'roman_urdu': assistant_text, 'arabic_urdu': assistant_text}

In [7]:
# 4) TTS: synthesize Urdu (Arabic script) and play/save it
def synthesize_urdu_to_mp3(urdu_text, out_path='response_urdu.mp3', lang='ur'):
    t = gTTS(text=urdu_text, lang=lang)
    t.save(out_path)
    display(Audio(out_path, autoplay=False))
    return out_path

In [ ]:
audio_file = 'tuition_fee.ogg'

urdu_script = transcribe_audio(audio_file)
print('Urdu Script:', urdu_script)

roman_question = urdu_to_roman(urdu_script)
print('Roman-Urdu:', roman_question)

resp = generate_response(roman_question)
print('Roman-Urdu Response:', resp.get('roman_urdu'))
# print('Urdu-Script Response:', resp.get('arabic_urdu'))

if resp.get('arabic_urdu'):
    synthesize_urdu_to_mp3(resp.get('arabic_urdu'))

Urdu Script: فاسٹ یونیورسٹی کی ٹیوشن فیس کتنی ہے؟
Roman-Urdu: Fast University ki tuition fees kitni hai?


PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\-\AppData\Local\Temp\ipykernel_7996\2449173277.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vect = Chroma(embedding_function=emb, persist_directory=chroma_persist_dir, collection_name=collection_name)


In [9]:
import glob

input_dir = 'Test_cases_inputs'
output_dir = 'test_results'
summary_file = os.path.join(output_dir, 'summary.json')
results = []

ogg_files = glob.glob(os.path.join(input_dir, '*.ogg'))
print(f"Found {len(ogg_files)} files to process.")

for audio_file in ogg_files:
    print(f"\nProcessing: {audio_file}")
    try:
        # 1. Transcribe
        urdu_script = transcribe_audio(audio_file)

        # 2. Romanize
        roman_question = urdu_to_roman(urdu_script)

        # 3. Query
        resp = generate_response(roman_question)
        roman_urdu_response = resp.get('roman_urdu', '')
        arabic_urdu_response = resp.get('arabic_urdu', '')

        # 4. Synthesize Voice
        out_filename = f"response_{os.path.basename(audio_file).replace('.ogg', '.mp3')}"
        out_path = os.path.join(output_dir, out_filename)

        if arabic_urdu_response:
            synthesize_urdu_to_mp3(arabic_urdu_response, out_path=out_path)

        results.append({
            'input_audio': audio_file,
            'input_roman': roman_question,
            'response_roman': roman_urdu_response,
            'response_audio': out_path
        })
        print(f"Success: {out_filename}")

    except Exception as e:
        print(f"Error processing {audio_file}: {e}")

# Save summary
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"\nBatch processing complete. Summary saved to {summary_file}")


Found 8 files to process.

Processing: Test_cases_inputs\input (1).ogg


Success: response_input (1).mp3

Processing: Test_cases_inputs\input (2).ogg


Success: response_input (2).mp3

Processing: Test_cases_inputs\input (3).ogg


Success: response_input (3).mp3

Processing: Test_cases_inputs\input (4).ogg


Success: response_input (4).mp3

Processing: Test_cases_inputs\input (5).ogg


Success: response_input (5).mp3

Processing: Test_cases_inputs\input (6).ogg


Success: response_input (6).mp3

Processing: Test_cases_inputs\input (7).ogg


Success: response_input (7).mp3

Processing: Test_cases_inputs\input (8).ogg


Success: response_input (8).mp3

Batch processing complete. Summary saved to test_results\summary.json


In [10]:

test_query = "Fast University ki tuition fees kitni hai?"

resp = generate_response(test_query)
roman_urdu_response = resp.get('roman_urdu', '')
arabic_urdu_response = resp.get('arabic_urdu', '')

print(roman_urdu_response)

if arabic_urdu_response:
    synthesize_urdu_to_mp3(arabic_urdu_response, out_path='test_single_response.mp3')

FAST University ki tuition fees Rs. 11000 per credit hour hai.
